# 00 — Define a finance evaluation success contract

**Goal:** distinguish correct numbers from an acceptable answer.

**Mode:** offline, synthetic, authored candidates. No model/API calls, SQL, MCP, or automated semantic grading occur here. We use a supplied demo review to show how grounding affects task completion.

By the end you can explain the input, reference, per-check verdict, and overall verdict. Next lesson adds actual database fixtures and independent reference calculations.

In [ ]:
import sys
from pathlib import Path
import json
from decimal import Decimal
from finance_evals.contracts import evaluate, percentage_change
print("Kernel:", sys.executable)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project:", ROOT)

## 1. The user journey

“Why did Marketplace revenue decline last month?”

Fix today to November 18, 2026. Compare October with September, in USD. Monthly totals support a change calculation, but do not establish churn as a cause. Success in this limited-evidence case means reporting the verified change and explicitly disclosing the causal limitation. This does not constitute a complete root-cause investigation.

In [ ]:
case = json.loads((ROOT / "evals/reference/revenue_case.json").read_text())
reference = case["reference"]
reference

## 2. Verify the arithmetic yourself

Amounts are integer cents. $10 million to $9 million means a $1 million decrease, divided by the prior $10 million: −10%. The displayed percentage is stored as a two-decimal string in this first contract. Later graders will use explicit tolerances. A zero denominator is undefined, not zero percent.

In [ ]:
delta = reference["current_cents"] - reference["prior_cents"]
pct = percentage_change(reference["current_cents"], reference["prior_cents"])
assert delta == -100_000_000
assert pct == Decimal("-10")
assert percentage_change(100, 0) is None
print("Change in USD:", Decimal(delta) / 100, "Percentage:", pct)

## 3. Predict the outcomes before running

The first answer reports the change and evidence limit. The second adds an unsupported churn claim. The third has an incorrect structured reporting period. Which should pass? Which individual checks should fail?

In [ ]:
answers = json.loads((ROOT / "evals/cases/lesson00_answers.json").read_text())
reviews = json.loads((ROOT / "evals/reviews/lesson00_reviews.json").read_text())
for answer in answers:
    print(answer["id"], "\n", answer["narrative"], "\nPeriod:", answer["current_period"], "\n")

## 4. Inspect the implementation

The evaluator compares expected fields, then consumes an explicit narrative review. The review is bound to the exact answer ID and narrative to detect stale reviews. It is a teaching input, not a trusted identity/signature mechanism. Arbitrary prose cannot be verified by these equality checks. Later we will validate narrative claims with a calibrated judge and evidence.

In [ ]:
import inspect
print(inspect.getsource(evaluate))

In [ ]:
reports = [evaluate(answer, reference, review) for answer, review in zip(answers, reviews)]
for report in reports:
    print(json.dumps(report, indent=2))
assert [r["overall"] for r in reports] == ["pass", "fail", "fail"]
assert reports[1]["checks"]["numbers_correct"] == "pass"
assert reports[1]["checks"]["claims_supported"] == "fail"

## 5. Missing evidence must stay visible

An absent or stale narrative review is **unverifiable**. Correct numbers do not turn missing evidence into a pass. Explicit failures remain failures even when another check is unverifiable.

In [ ]:
unreviewed = evaluate(answers[0], reference)
assert unreviewed["overall"] == "unverifiable"
edited = dict(answers[0], narrative="Revenue rose by 20%.")
assert evaluate(edited, reference, reviews[0])["checks"]["claims_supported"] == "unverifiable"
unreviewed

## 6. Interpret what we established

We demonstrated a success contract and aggregation of per-check results on three deliberately authored examples. This is not an agent success-rate estimate. Numerical correctness, scope correctness, and grounded explanation remain separate requirements.

**Checkpoint:** Why does the churn example fail despite matching all reference figures? What additional evidence would be needed to support a churn contribution?

**Next:** create source records, a database snapshot, independent reference queries, and golden-case review records in Notebook 01.